In [2]:
import re
import sys
from pathlib import Path
import fitz
import chromadb
from sentence_transformers import SentenceTransformer

In [3]:
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

In [4]:
PDF_DIR = Path("../data/agriculture")
TXT_DIR = Path("data/extracted")
CHROMA_DIR = Path("data/chroma_db")
COLLECTION_NAME = "agriculture_manuals"

In [5]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
EMBEDDING_MODEL = "paraphrase-multilingual-mpnet-base-v2"

In [6]:
ocr_reader = None

def get_ocr_reader():
    global ocr_reader
    if ocr_reader is None:
        try:
            import easyocr
            ocr_reader = easyocr.Reader(["pt"], gpu=False)
        except ImportError:
            raise ImportError(
                "easyocr is not installed. Please install it to use OCR functionality."
            )
    return ocr_reader

In [7]:
def extract_with_ocr(pdf_path: Path) -> str:
    reader = get_ocr_reader()
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc):
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))  # Aumenta a resolução para melhor OCR
        img = pix.tobytes("png")
        results = reader.readtext(img, detail=0, paragraph=True)
        page_text = "\n".join(results).strip()
        if page_text:
            pages.append(page_text)
        print(f" [OCR] Page {i + 1}/{len(doc)} processed.")
    doc.close()
    return "\n\n".join(pages)

In [8]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        text = page.get_text("text")
        if text.strip():
            pages.append(text)
    doc.close()
    text = "\n\n".join(pages)
    
    if len(text.strip()) <= 0:
        print(f" [INFO] No text found in {pdf_path}. Attempting OCR...")
        text = extract_with_ocr(pdf_path)

    return text

In [9]:
def clean_text(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" {2,}", " ", text)
    text = re.sub(r"\x00", "", text)
    return text.strip()

In [10]:
def chunk_text(text: str) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + CHUNK_SIZE
        chunk = text[start:end]
        if end < len(text):
            boundary = max(
                chunk.rfind(". "),
                chunk.rfind(".\n"),
                chunk.rfind("! "),
                chunk.rfind("? "),
                chunk.rfind("\n\n")
            )
            if boundary > CHUNK_SIZE // 2:
                end = start + boundary + 1
                chunk = text[start:end]
        
        chunk = chunk.strip()
        if chunk:
            chunks.append(chunk)
        start = end - CHUNK_OVERLAP

    return chunks

In [11]:
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
if not pdf_files:
    print(f"No PDF file found in {PDF_DIR}")
    sys.exit(1)

print(f"Found {len(pdf_files)} PDFs\n")

Found 24 PDFs



In [12]:
all_chunks: list[str] = []
all_metadata: list[dict] = []
all_ids: list[str] = []

TXT_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

for pdf_path in pdf_files:
    print(f"Extracting: {pdf_path.name}")
    try:
        raw_text = extract_text_from_pdf(pdf_path)
        text = clean_text(raw_text)
    except Exception as e:
        print(f"Error extracting text from {pdf_path.name}: {e}")
        continue
    txt_path = TXT_DIR / f"{pdf_path.stem}.txt"
    txt_path.write_text(text, encoding="utf-8")
    print(f"Saved extracted text to: {txt_path}\n")

    chunks = chunk_text(text)
    print(f"Created {len(chunks)} chunks for {pdf_path.name}\n")

    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_metadata.append({
            "source": pdf_path.name,
            "chunk_index": i,
            "total_chunks": len(chunks)
        })
        all_ids.append(f"{pdf_path.stem}_chunk_{i}")

print(f"Total chunks created: {len(all_chunks)}\n")

Extracting: 5.-Rec.-Agr.-Maputo.pdf
Saved extracted text to: data\extracted\5.-Rec.-Agr.-Maputo.txt

Created 5 chunks for 5.-Rec.-Agr.-Maputo.pdf

Extracting: AF_Catalogo-Variedades.pdf


Using CPU. Note: This module is much faster with a GPU.


Saved extracted text to: data\extracted\AF_Catalogo-Variedades.txt

Created 34 chunks for AF_Catalogo-Variedades.pdf

Extracting: Amaranthus.pdf
 [INFO] No text found in ..\data\agriculture\Amaranthus.pdf. Attempting OCR...


c:\Users\user\OneDrive\amigo_do_agricultor_project\code\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


 [OCR] Page 1/2 processed.
 [OCR] Page 2/2 processed.
Saved extracted text to: data\extracted\Amaranthus.txt

Created 8 chunks for Amaranthus.pdf

Extracting: Farinha-de-feijao-jugo.pdf
 [INFO] No text found in ..\data\agriculture\Farinha-de-feijao-jugo.pdf. Attempting OCR...
 [OCR] Page 1/8 processed.
 [OCR] Page 2/8 processed.
 [OCR] Page 3/8 processed.
 [OCR] Page 4/8 processed.
 [OCR] Page 5/8 processed.
 [OCR] Page 6/8 processed.
 [OCR] Page 7/8 processed.
 [OCR] Page 8/8 processed.
Saved extracted text to: data\extracted\Farinha-de-feijao-jugo.txt

Created 5 chunks for Farinha-de-feijao-jugo.pdf

Extracting: Folheto_Inhame.pdf
 [INFO] No text found in ..\data\agriculture\Folheto_Inhame.pdf. Attempting OCR...
 [OCR] Page 1/2 processed.
 [OCR] Page 2/2 processed.
Saved extracted text to: data\extracted\Folheto_Inhame.txt

Created 8 chunks for Folheto_Inhame.pdf

Extracting: Madioca-Variedade-_ORERA.pdf
 [INFO] No text found in ..\data\agriculture\Madioca-Variedade-_ORERA.pdf. Attem

In [13]:
len(all_chunks)

579

In [14]:
model = SentenceTransformer(EMBEDDING_MODEL)

print(f"Generating embeddings...")
embeddings = model.encode(
    all_chunks,
    show_progress_bar=True,
    batch_size=32,
    convert_to_numpy=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generating embeddings...


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

In [15]:
print(len(all_chunks))

579


In [16]:
client = chromadb.PersistentClient(path=CHROMA_DIR)

try:
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection: {COLLECTION_NAME}")
except Exception as e:
    print(f"Error occurred while deleting collection: {e}")
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

batch_size = 100
for i in range(0, len(all_chunks), batch_size):
    batch_end = min(i + batch_size, len(all_chunks))
    batch_ids = all_ids[i:batch_end]
    batch_embeddings = embeddings[i:batch_end].tolist()
    batch_metadata = all_metadata[i:batch_end]
    batch_documents = all_chunks[i:batch_end]

    collection.add(
        ids=batch_ids,
        embeddings=batch_embeddings,
        metadatas=batch_metadata,
        documents=batch_documents
    )
    print(f"Added chunks {i} to {batch_end} to the collection.")

    total = collection.count()
    print(f"Finished! Total chunks in collection {COLLECTION_NAME}: {total}\n")
    print(f"Vector database is in {CHROMA_DIR.resolve()}\n")

Deleted existing collection: agriculture_manuals
Added chunks 0 to 100 to the collection.
Finished! Total chunks in collection agriculture_manuals: 100

Vector database is in C:\Users\user\OneDrive\amigo_do_agricultor_project\code\notebooks\data\chroma_db

Added chunks 100 to 200 to the collection.
Finished! Total chunks in collection agriculture_manuals: 200

Vector database is in C:\Users\user\OneDrive\amigo_do_agricultor_project\code\notebooks\data\chroma_db

Added chunks 200 to 300 to the collection.
Finished! Total chunks in collection agriculture_manuals: 300

Vector database is in C:\Users\user\OneDrive\amigo_do_agricultor_project\code\notebooks\data\chroma_db

Added chunks 300 to 400 to the collection.
Finished! Total chunks in collection agriculture_manuals: 400

Vector database is in C:\Users\user\OneDrive\amigo_do_agricultor_project\code\notebooks\data\chroma_db

Added chunks 400 to 500 to the collection.
Finished! Total chunks in collection agriculture_manuals: 500

Vector 

In [17]:
print(client.list_collections())

[Collection(name=agriculture_manuals)]


In [19]:
col = client.get_collection(COLLECTION_NAME)
print(f"Number of items in collection '{COLLECTION_NAME}': {col.count()}")

Number of items in collection 'agriculture_manuals': 579


In [20]:
col.peek(5)

{'ids': ['5.-Rec.-Agr.-Maputo_chunk_0',
  '5.-Rec.-Agr.-Maputo_chunk_1',
  '5.-Rec.-Agr.-Maputo_chunk_2',
  '5.-Rec.-Agr.-Maputo_chunk_3',
  '5.-Rec.-Agr.-Maputo_chunk_4'],
 'embeddings': array([[-0.17952123,  0.01417133, -0.010572  , ...,  0.04370495,
         -0.00904416, -0.0558057 ],
        [-0.08847558, -0.036236  , -0.01679771, ...,  0.00877946,
         -0.0208837 , -0.00612334],
        [-0.11414379,  0.0057024 , -0.01122835, ...,  0.02363853,
         -0.02354028,  0.01183464],
        [-0.1005221 ,  0.07100505, -0.01097958, ...,  0.00484916,
         -0.02124477,  0.02233628],
        [-0.02208672,  0.11538103, -0.00873387, ...,  0.09342187,
          0.00047935, -0.03742265]], shape=(5, 768)),
 'documents': ['25ºS\n24ºS\n26ºS\n30ºS\n30ºS\n30ºS\n27ºS\n30ºE\n30º\n30º\n30º\n31ºE\n32ºE\n33ºE\nPREVISÃO DA PRECIPITAÇÃO\nPREVISÃO CLIMÁTICA DA PRECIPITAÇÃO: JFM 2023 \nNormal para acima do normal\nClimatologia (Normal)\nNormal para baixo do normal\nAbaixo do normal para normal\nPREV

In [29]:
query_embedding = model.encode(["Fale sobre o Inhame?"], convert_to_numpy=True)

result = col.query(query_embeddings=query_embedding.tolist())

for doc, metadata in zip(result['documents'][0], result['metadatas'][0]):
    print(f"--- {metadata}")
    print(doc)

--- {'total_chunks': 288, 'chunk_index': 195, 'source': 'MANUAL-DE-MILHO.pdf'}
nosclerospora sorghi)
É uma doença de extrema importância económica no país, principalmente nas zonas de alta humidade e com 
períodos de temperaturas amenas (Casela, Ferreira, & Silva e Pinto, 2006). Em Moçambique, nas províncias de 
Maputo, Gaza e parte de Inhambane, durante muito tempo era encontrada a doença em abundância durante 
a época fresca. Na região centro e norte não era frequente encontrar a doença até a alguns anos atrás, mas 
ultimamente já se nota a sua distribuição em locais que não apresentavam a doença.
Os sintomas característicos são o amarelecimento progressivo desde a base da folha (lígula) para o topo e 
o aparecimento de esporos em forma de pó branco na superfície do limbo, principalmente na página inferior 
(Figura 2). Plantas infestadas precocemente morrem antes de florir.
--- {'chunk_index': 16, 'total_chunks': 26, 'source': 'Manual-de-Producao-COGUMELOS.pdf'}
. E 
necessário fumig